# Exercise P12.1: TF-IDF Text Classification
### STAT 540 — Week 12


## Overview

Build a TF-IDF + logistic regression pipeline to classify movie reviews as positive or negative.

## Task 1: Load and Split Data

In [5]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

df = load_dataset("stanfordnlp/imdb", split="train").to_pandas()

# Clean once, here, so the TF-IDF model and the LLM see the exact same text
df["text"] = (df["text"]
              .str.replace(r"<br\s*/?>", " ", regex=True)
              .str.replace(r"\s+", " ", regex=True)
              .str.strip())

# Shuffle before sampling because the raw split is sorted by label
df = df.sample(5000, random_state=42).reset_index(drop=True)
texts, labels = df["text"].tolist(), df["label"].tolist()   # 0 = neg, 1 = pos

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 4000, Test: 1000


Task 2: Build and Evaluate Pipeline

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000))
])
pipe.fit(X_train, y_train)

print(f"Test Accuracy: {pipe.score(X_test, y_test):.3f}")
print(classification_report(y_test, pipe.predict(X_test)))

Test Accuracy: 0.861
              precision    recall  f1-score   support

           0       0.89      0.83      0.86       503
           1       0.84      0.89      0.86       497

    accuracy                           0.86      1000
   macro avg       0.86      0.86      0.86      1000
weighted avg       0.86      0.86      0.86      1000



In [7]:
import joblib
joblib.dump(pipe, "tfidf_pipe.joblib")
pd.DataFrame({"text": X_test, "label": y_test}).to_csv("imdb_test_split.csv", index=False)

## Task 3: Inspect Most Informative Words

In [8]:
import pandas as pd
feature_names = pipe.named_steps["tfidf"].get_feature_names_out()
coefs = pipe.named_steps["clf"].coef_[0]

coef_df = pd.DataFrame({"word": feature_names, "coef": coefs})
print("Most POSITIVE:\n", coef_df.nlargest(10, "coef")[["word", "coef"]].to_string(index=False))
print("\nMost NEGATIVE:\n", coef_df.nsmallest(10, "coef")[["word", "coef"]].to_string(index=False))

Most POSITIVE:
      word     coef
    great 3.876077
excellent 2.795252
     best 2.671763
     love 2.474052
wonderful 2.335116
    today 2.020032
      fun 2.006654
    loved 1.826599
  enjoyed 1.815485
  amazing 1.792868

Most NEGATIVE:
     word      coef
     bad -4.700423
   worst -4.532629
terrible -2.631883
   awful -2.576690
   waste -2.520170
  boring -2.449892
    poor -2.437100
  script -2.175346
   worse -2.162992
  better -2.076071


**Your turn:** Do the most informative words make sense? Any surprises?

> *I think they make a lot of sense. The words for both the positive and negative descriptions accurately express the sentiment by using adjectives. I do think it is interesting that the word "script" is present in the negative words list, and that nothing characteristic about the movie (like the word plot) is present.*

## Submission

```bash
git add week12/exercises/P12.1*
git commit -m "Complete Exercise P12.1: TF-IDF text classification"
git push origin main
```